# Week 7, Day 1: Default and keyword arguments

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w007d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def greet(name, greeting="Hello", punct="!"):
    return f"{greeting}, {name}{punct}"

print(greet("Ada"))
print(greet("Ada", punct="?"))
print(greet(greeting="Hi", name="Bo"))

In [ ]:
rate = 0.05

def with_interest(amount, r=rate):
    return amount * (1 + r)

print(with_interest(1000))
rate = 0.10
print(with_interest(1000))

> **Key idea.** Default values are evaluated **once**, when the function is defined. They are not recalculated on each call.

In [ ]:
def add_item(item, basket="".split()):
    basket.append(item)
    return basket

print(add_item("rice"))
print(add_item("beans"))

In [ ]:
def with_interest(amount, r=None):
    if r is None:
        r = current_rate()
    return amount * (1 + r)

def current_rate():
    return 0.07

print(with_interest(1000))
print(with_interest(1000, 0.02))

> **Tip.** Check for "not given" with `is None`, not with `if not r:`. A caller might pass `0` on purpose, and `0` is falsy too.

In [ ]:
def make_rectangle(width, height, *, filled=False, char="#"):
    for _ in range(height):
        print(char * width if filled else char + " " * (width - 2) + char)

make_rectangle(6, 3, filled=True)
make_rectangle(6, 3, char="+")

In [ ]:
def make_rectangle(width, height, *, filled=False, char="#"):
    pass

make_rectangle(6, 3, True)

> **Key idea.** Put a bare `*` in a parameter list to make every parameter after it keyword-only. Use it for flags (`True`/`False` options) and for settings that would be confusing as bare values.

In [ ]:
help(abs)

In [ ]:
def distance(x1, y1, x2, y2, /):
    return ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5

print(distance(0, 0, 3, 4))

In [ ]:
def format_price(amount, *, symbol="₦", places=2, show_sign=False):
    """Return amount formatted as money, like '₦1,234.50'."""
    sign = "+" if show_sign and amount > 0 else ""
    return f"{sign}{symbol}{amount:,.{places}f}"

print(format_price(1234.5))
print(format_price(99, symbol="$", places=0))
print(format_price(250, show_sign=True))

### Exercise: When was it worked out?

Predict what this prints, then fill in your answers.

```python
size = 3

def stars(n=size):
    return "*" * n

size = 5
print(stars())      # line 1
print(stars(size))  # line 2
```

In [ ]:
line_1 = "___"
line_2 = "___"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"both lines\"\n    assert line_1 == \"***\", \"The default was worked out when def ran, when size was 3.\"\n    assert line_2 == \"*****\", \"Here size is passed at the call, when it's 5.\"")

### Exercise: None means not given

Write `greeting(name, hour=None)` that returns `Good morning, NAME` if the hour is below 12, `Good afternoon, NAME` if it's below 18, and `Good evening, NAME` otherwise. If no hour is given, use the value returned by `current_hour()` (already written). An hour of `0` is midnight, a real hour, so check with `is None`.

In [ ]:
def current_hour():
    return 15

def greeting(name, hour=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_given():\n    \"with an hour\"\n    assert greeting(\"Ada\", 9) == \"Good morning, Ada\"\n    assert greeting(\"Ada\", 12) == \"Good afternoon, Ada\"\n    assert greeting(\"Ada\", 20) == \"Good evening, Ada\"\n    assert greeting(\"Ada\", 0) == \"Good morning, Ada\", \"0 is midnight: check with 'is None', not 'if not hour'.\"\ndef test_default():\n    \"without an hour\"\n    assert greeting(\"Bo\") == \"Good afternoon, Bo\", \"With no hour, use current_hour().\"")

### Exercise: Name your flags

Change the `def` line of `describe` so that `shout` and `excited` are **keyword-only**, without changing the body. Then fix the last line so it still prints `HELLO THERE!`.

In [ ]:
def describe(text, shout=False, excited=False):
    if shout:
        text = text.upper()
    if excited:
        text += "!"
    return text

print(describe("hello there", True, True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints HELLO THERE!\"\n    expect_output(\"HELLO THERE!\")\ndef test_keyword_only():\n    \"flags can't be passed by position\"\n    try:\n        describe(\"x\", True)\n    except TypeError:\n        pass\n    else:\n        assert False, \"describe('x', True) should be refused: put a * before shout.\"")

### Exercise: Positional only

Write `hypotenuse(a, b, /)` returning the length of the longest side of a right-angled triangle with sides `a` and `b`, rounded to 2 places. Both parameters must be positional-only.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several triangles\"\n    assert hypotenuse(3, 4) == 5.0\n    assert hypotenuse(1, 1) == 1.41\ndef test_posonly():\n    \"can't be called with names\"\n    try:\n        hypotenuse(a=3, b=4)\n    except TypeError:\n        pass\n    else:\n        assert False, \"hypotenuse(a=3, b=4) should be refused: put a / after the parameters.\"")

### Exercise: A friendly signature

Write `format_price(amount, *, symbol="₦", places=2, show_sign=False)` that returns the amount formatted as money: an optional `+` (only when `show_sign` is True and the amount is above 0), the symbol, then the amount with thousands separators and `places` decimals. Negative amounts put the minus before the symbol: `-₦5.00`.

In [ ]:
def format_price(amount, *, symbol="₦", places=2, show_sign=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several prices\"\n    assert format_price(1234.5) == \"\u20a61,234.50\"\n    assert format_price(99, symbol=\"$\", places=0) == \"$99\"\n    assert format_price(250, show_sign=True) == \"+\u20a6250.00\"\n    assert format_price(-5) == \"-\u20a65.00\"\n    assert format_price(-5, show_sign=True) == \"-\u20a65.00\"\n    assert format_price(0, show_sign=True) == \"\u20a60.00\"\ndef test_kwonly():\n    \"the options are keyword-only\"\n    try:\n        format_price(1, \"$\")\n    except TypeError:\n        pass\n    else:\n        assert False, \"Keep the * so the options must be named.\"")

### Exercise: Redesign the call

This function is called with a mystery list of values. Redesign its `def` line so that `width` and `height` come first and are required, and `border`, `fill` and `title` are keyword-only with defaults `"#"`, `" "` and `""`. Then rewrite the two calls to match, so the output is unchanged.

In [ ]:
def panel(border, fill, title, width, height):
    print(border * width)
    inner = title.center(width - 2, fill) if title else fill * (width - 2)
    print(border + inner + border)
    for _ in range(height - 3):
        print(border + fill * (width - 2) + border)
    print(border * width)

panel("#", " ", "", 8, 4)
panel("*", ".", "Hi", 10, 5)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"same output\"\n    expect_output(\"########\\n#      #\\n#      #\\n########\\n**********\\n*...Hi...*\\n*........*\\n*........*\\n**********\")\ndef test_signature():\n    \"the new signature\"\n    import inspect\n    ps = inspect.signature(panel).parameters\n    names = list(ps)\n    assert names[:2] == [\"width\", \"height\"], \"width and height should come first.\"\n    for n, d in [(\"border\", \"#\"), (\"fill\", \" \"), (\"title\", \"\")]:\n        assert ps[n].kind is inspect.Parameter.KEYWORD_ONLY and ps[n].default == d, f\"{n} should be keyword-only with default {d!r}.\"")

### Bug hunt: Zero is not nothing

`repeat("ab", 0)` should return an empty string, but it returns `"ab"`. The default is meant to be "once". Fix the check.

In [ ]:
def repeat(text, times=None):
    if not times:
        times = 1
    return text * times

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several calls\"\n    assert repeat(\"ab\") == \"ab\"\n    assert repeat(\"ab\", 3) == \"ababab\"\n    assert repeat(\"ab\", 0) == \"\", \"0 is a real value. Check for None with 'is None'.\"")

### Bug hunt: Stuck in the past

`stamp()` should label each message with the **current** version, but after `VERSION` changes it still uses the old one. Fix the function so the default is worked out at each call.

In [ ]:
VERSION = "1.0"

def stamp(message, version=VERSION):
    return f"[v{version}] {message}"

print(stamp("started"))
VERSION = "2.0"
print(stamp("updated"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"uses the current version\"\n    expect_output(\"[v1.0] started\\n[v2.0] updated\")")

### Bug hunt: A refused call

This crashes with a `TypeError`. The function is designed correctly; fix the call so it prints `$12.50`.

In [ ]:
def price(amount, *, symbol="₦"):
    return f"{symbol}{amount:.2f}"

print(price(12.5, "$"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints $12.50\"\n    expect_output(\"$12.50\")\ndef test_def():\n    \"the function is unchanged\"\n    assert 'def price(amount, *, symbol=\"\u20a6\"):' in source")

### Puzzle: An optional prefix

Arrange the lines into a function with an optional prefix that's worked out at call time. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Which calls work?

For `def f(a, /, b, *, c=0):`, say whether each call works. Write `True` if it runs, `False` if it raises a `TypeError`.

In [ ]:
call_1 = ___   # f(1, 2)
call_2 = ___   # f(1, b=2)
call_3 = ___   # f(a=1, b=2)
call_4 = ___   # f(1, 2, 3)
call_5 = ___   # f(1, 2, c=3)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_calls():\n    \"each call\"\n    assert call_1 is True, \"a by position, b by position: fine.\"\n    assert call_2 is True, \"b may be named.\"\n    assert call_3 is False, \"a is positional-only.\"\n    assert call_4 is False, \"c is keyword-only, so 3 has nowhere to go.\"\n    assert call_5 is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a table formatter

Write `table_row(*, cells, widths, align="left", sep=" | ")` that returns one row of a text table, where `cells` is a string of values separated by commas, and `widths` is a string of column widths separated by commas. Each value is stripped, cut to its column's width if it's too long, and aligned within it: `align` is `"left"`, `"right"` or `"centre"` for every column. The cells are joined by `sep`.

Every parameter is keyword-only, so calls look like this:

```python
table_row(cells="Ada, 93, A", widths="8,4,2")                 # 'Ada      | 93   | A '
table_row(cells="Ada, 93, A", widths="8,4,2", align="right")  # '     Ada |   93 |  A'
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def row(cells, widths, align=\"left\", sep=\" | \"):\n    out = []\n    for c, w in zip(cells.split(\",\"), widths.split(\",\")):\n        c, w = c.strip()[:int(w)], int(w)\n        out.append(c.ljust(w) if align == \"left\" else c.rjust(w) if align == \"right\" else c.center(w))\n    return sep.join(out)\ndef test_values():\n    \"several rows\"\n    for kw in [dict(cells=\"Ada, 93, A\", widths=\"8,4,2\"), dict(cells=\"Ada, 93, A\", widths=\"8,4,2\", align=\"right\"),\n               dict(cells=\"Grace Hopper, 100\", widths=\"5,5\", align=\"centre\"), dict(cells=\"x,y,z\", widths=\"3,3,3\", sep=\"\")]:\n        assert table_row(**kw) == row(**kw), f\"table_row({kw}) should be {row(**kw)!r}.\"\ndef test_keyword_only():\n    \"every parameter is keyword-only\"\n    try:\n        table_row(\"a\", \"1\")\n    except TypeError:\n        pass\n    else:\n        assert False, \"Start the parameter list with * so every argument must be named.\"")

## Warm-up

All from last week. Today's question: how far can you push this, and where are the traps?

## When defaults are worked out

A default value is worked out **once**, when Python runs the `def` line, and that same value is used for every call that leaves the argument out. Most of the time you'd never notice. Here's a case where you would:

Both lines print 1050.0. When the `def` ran, `rate` was 0.05, so the default became 0.05, and changing `rate` afterwards doesn't touch it. Press X-ray and look at the function object: the default is stored inside it.

If you wanted the **current** rate at every call, the function should read the global (or better, take it as an argument) inside its body, not in its `def` line.

### Why this matters: defaults that can change

For numbers and strings, which can't be changed, being worked out once is harmless. For values that **can** change in place, like the lists you'll meet next week, it's a famous trap: every call shares the same object. Here's a preview, using `.split()` to make a list and `.append()` to add to it:

The second call still has `"rice"` in it! There's only one list, made when the `def` ran, and both calls add to it. You'll study this properly in Week 8. The rule to remember now: **never use a changeable value as a default**. Use `None` instead.

## None as a default

When a default must be worked out at call time, or must be a fresh object each time, use `None` to mean "the caller didn't give one", and fill in the real value inside the function:

This pattern is everywhere in Python's own libraries. When you read `x=None` in a signature, it almost always means "optional; if you don't give it, something sensible is worked out for you".

## Keyword-only parameters

Look at this call:

```python
make_rectangle(10, 4, True, False)
```

What do `True` and `False` mean? Nobody can tell without looking up the function. If those parameters had to be **named** in the call, the code would explain itself:

```python
make_rectangle(10, 4, filled=True, rounded=False)
```

You can insist on that. Parameters after a bare `*` in the `def` are **keyword-only**: they can only be given by name.

`takes 2 positional arguments but 3 were given`: the `True` can't sneak into `filled` by position.

## Positional-only parameters

The opposite also exists. Parameters **before** a `/` are **positional-only**: they can't be given by name. You'll mostly see this in Python's documentation for built-in functions:

`abs(x, /)` means you write `abs(-5)`, never `abs(x=-5)`. Built-ins do this so that the parameter's name isn't part of the promise: it could be renamed without breaking anyone's code. You can use it too:

### Reading signatures in the documentation

You can now read any signature in Python's documentation. For example, `help(print)` shows:

```
print(*objects, sep=' ', end='\n', file=None, flush=False)
```

- `*objects`: any number of values to print. (That `*` before a name is tomorrow's lesson.)
- Everything after it is keyword-only, because it comes after a `*`: you must write `sep="-"`, never a bare `"-"`.
- `file=None`: the "not given" pattern. If you don't give a file, `print` uses the screen.

## Designing a good parameter list

A function's parameters are its **interface**: the part other people (and you, next month) have to get right every time they call it. A few guidelines:

1. **Most important first.** The values every call needs come first, in the order people would naturally say them: `area(width, height)`, `send(to, subject, body)`.
2. **Required before optional.** Python forces this anyway.
3. **Flags are keyword-only.** `True` and `False` mean nothing on their own in a call.
4. **Safe defaults.** A default should be the value that's right most of the time, and never a changeable object.
5. **Not too many.** More than four or five parameters usually means the function is doing several jobs and should be split.

Every call reads clearly, and nobody can accidentally pass a currency symbol into the `places` slot.

## Practice

## Quiz

## Daily challenge

## Before you go

Write down from memory what `*` and `/` mean in a parameter list, and when a default value is worked out. Then look up `help(sorted)` and `help(str.split)` and read their signatures: you can now understand every part. Tomorrow: `*args` and `**kwargs`, the stars that let a function take any number of arguments.